### Connect with the Google Drive

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
!pip install --upgrade neurokit2
!pip install --upgrade mne
!pip install mne-icalabel

### Import libraries

In [ ]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import seaborn as sns
import os
import mne
from scipy.io import loadmat
from scipy import signal
from matplotlib import pyplot as plt
from mne.preprocessing import ICA
from mne_icalabel import label_components
from sklearn.model_selection import LeaveOneGroupOut
from scipy.stats.mstats import winsorize


### Load data file

In [ ]:
def load_data_file(dir_path, file_name):
    if not os.path.exists(dir_path):
        print("Directory does not exist. Creating it.")
        os.makedirs(dir_path)

    data_path = os.path.join(dir_path, file_name)
    if not os.path.exists(data_path):
        print("File does not exist. Please check the file name.")

    raw_data = loadmat(data_path)
    dreamer_data = raw_data['DREAMER'] #DREAMER is inside of the raw_data, the size of the array is 1
    dreamer_raw = dreamer_data[0, 0] #EEG_sampling_rate, ECG_SamplingRate, EEG_Electrodes, Data, noOfSubjects, noOfVideoSequences are inside of the dreamer_raw
    dreamer_raw_data = dreamer_raw['Data']

    return dreamer_raw_data, dreamer_raw

### Extract the number of participants and video clips

In [ ]:
def extract_participants_and_clips(dreamer_raw):
    participants = dreamer_raw['noOfSubjects']
    num_participants = participants[0, 0]
    video_clips = dreamer_raw['noOfVideoSequences']
    num_video_clips = video_clips[0, 0]

    return num_participants, num_video_clips


### Extract physiological signal information in the dataset

In [ ]:
def extract_signal_information(dreamer_raw):
    eeg_sample_rate = dreamer_raw['EEG_SamplingRate']
    eeg_rate = eeg_sample_rate[0, 0]
    ecg_sample_rate = dreamer_raw['ECG_SamplingRate']
    ecg_rate = ecg_sample_rate[0, 0]
    electrodes = dreamer_raw['EEG_Electrodes']
    num_electrodes = len(electrodes[0])
    eeg_channels = []

    for ichannel in range(num_electrodes):
        eeg_channels.append(electrodes[0, ichannel][0].item())

    return eeg_rate, ecg_rate, eeg_channels

### Extract Valence, Arousal, Dominance values for one video watched by one participant

In [ ]:
def extract_vad_one_video_one_participant(dreamer_raw_data, iparticipant, ivideo):
    valence = dreamer_raw_data[0,iparticipant]['ScoreValence'][0, 0][ivideo, 0]
    arousal = dreamer_raw_data[0,iparticipant]['ScoreArousal'][0, 0][ivideo, 0]
    dominance = dreamer_raw_data[0,iparticipant]['ScoreDominance'][0, 0][ivideo, 0]

    return valence, arousal, dominance

### Extract Valence, Arousal, Dominance values for one participant

In [ ]:
def extract_vad_one_participant(dreamer_raw_data, iparticipant, num_video_clips):
    valences = []
    arousals = []
    dominances = []

    for ivideo in range(num_video_clips):
        valence, arousal, dominance = extract_vad_one_video_one_participant(dreamer_raw_data, iparticipant, ivideo)
        valences.append(valence)
        arousals.append(arousal)
        dominances.append(dominance)

    return valences, arousals, dominances

### Extract Valence, Arousal, Dominance values for all participants

In [ ]:
def extract_vad_all_participants(dreamer_raw_data, num_participants, num_video_clips):
  valences = []
  arousals = []
  dominances = []

  for iparticipant in range(num_participants):
    valence, arousal, dominance = extract_vad_one_participant(dreamer_raw_data, iparticipant, num_video_clips)
    valences.append(valence)
    arousals.append(arousal)
    dominances.append(dominance)

  return valences, arousals, dominances

### Extract all physiological signals for one participant

In [ ]:
def extract_physiological_signal_one_participant(dreamer_raw_data, iparticipant):
    #dreamer_raw_data[0] has 23 columns, each column represents a participant. Each participant has the following information: age, gender, eeg, ecg, vad score
    eeg_signal = dreamer_raw_data[0, iparticipant]['EEG']
    #The EEG signal of a participant contains the following information: baseline, stimuli
    eeg_baseline = eeg_signal[0, 0]['baseline']
    eeg_stimuli = eeg_signal[0, 0]['stimuli']

    ecg_signal = dreamer_raw_data[0, iparticipant]['ECG']
    ecg_baseline = ecg_signal[0, 0]['baseline']
    ecg_stimuli = ecg_signal[0, 0]['stimuli']

    return eeg_baseline, eeg_stimuli, ecg_baseline, ecg_stimuli

### Extract all physiological signals for all participants

In [ ]:
def extact_physiological_signal_all_participants(dreamer_raw_data, num_participants):

    eeg_baseline_all = []
    eeg_stimuli_all = []
    ecg_baseline_all = []
    ecg_stimuli_all = []


    for iparticipant in range(num_participants):
        eeg_baseline, eeg_stimuli, ecg_baseline, ecg_stimuli = extract_physiological_signal_one_participant(dreamer_raw_data, iparticipant)
        eeg_baseline_all.append(eeg_baseline)
        eeg_stimuli_all.append(eeg_stimuli)
        ecg_baseline_all.append(ecg_baseline)
        ecg_stimuli_all.append(ecg_stimuli)

    return eeg_baseline_all, eeg_stimuli_all, ecg_baseline_all, ecg_stimuli_all

### To plot the power spectral density (PSD) of the ICA components to help identify artifacts

In [ ]:
def plot_ica_psd_spectrum(ica, raw, atitle=None):
    # Plot Sources to identify artifacts
    # This opens an interactive window showing the IC time series
    ica.plot_sources(raw, title=atitle)
    # Compute the spectrum (fmax=60Hz is standard for DREAMER's 128Hz rate)
    spectrum = raw.compute_psd(fmax=60)

    #  Plot the PSD
    # 'spatial_colors=True' helps identify channel locations by colour
    spectrum.plot(picks='eeg', spatial_colors=True, average=False)


### Set up ICA for each video clip and display its information

In [ ]:
def set_up_ICA_labels_each_video(raw, eeg_channels):
    ica = ICA(n_components=len(eeg_channels), method='fastica', random_state=42)
    ica.fit(raw)

    #plot the ICA components to visually inspect them and identify artifacts (e.g. eye blinks, muscle activity, etc.)
   # ica.plot_components()

    #plot the power spectral density (PSD) of the ICA components to help identify artifacts (e.g. eye blinks have a characteristic low-frequency spectrum, muscle activity has a characteristic high-frequency spectrum, etc.)
    #plot_ica_psd_spectrum(ica, raw, atitle='ICA Components - DREAMER BEFORE REMOVAL')

    ica_labels = label_components(raw, ica, method='iclabel')
    ic_labels = ica_labels['labels']

    print(ic_labels) # the labels of the 14 components, e.g. 'brain', 'eye blink', 'muscle artifact', etc.


    return ica, ic_labels

### Identifiy and removal all artefacts with ICA in one video clip wtched by one participant

In [ ]:
def remove_artifacts_eeg_one_video(eeg_rate, eeg_stimuli, ivedio, eeg_channels):

    samples_one_video = eeg_stimuli[0, 0][ivedio, 0]

   # print(samples_one_video.shape) # the samples of all channels of one video clip

    # Apply ICA to remove artifacts with 14 channels (e.g eye blinks, muscle activity, etc.)
    info = mne.create_info(ch_names=eeg_channels, sfreq=eeg_rate, ch_types='eeg')
    raw = mne.io.RawArray(samples_one_video.T, info)

    # Set the 10-20 montage (Fixes the "Montage is not set" error)
    raw.set_montage('standard_1020')

    # Apply 1Hz high-pass filter (essential for ICA stability)
    raw.filter(l_freq=1.0, h_freq=None)

    #set up ICA and label components for one video clip
    ica, ic_labels = set_up_ICA_labels_each_video(raw, eeg_channels)

    # Identify indices of all non-brain (artifact) components
    ica.exclude = [i for i, label in enumerate(ic_labels) if label != 'brain']
    print(f"Excluding components: {ica.exclude}")

    # Apply the correction to the raw data
    raw_cleaned = ica.apply(raw.copy())

    removed_raw = mne.io.RawArray(raw_cleaned.get_data(), info)

    #plot the ICA components and their PSD spectrum after artifact removal to visually inspect the results
   # plot_ica_psd_spectrum(ica, removed_raw, atitle='ICA Components - DREAMER AFTER REMOVAL')

    return raw_cleaned.get_data().T



### Compute PSD and plot the frequencies against the PSD to confirm the further removal for EEG

In [ ]:
def plot_filter_psd_signal_eeg(original_signal, filter_signal, cleaned, eeg_detrended, eeg_rate):
    # Compute PSD
    psd_raw = nk.signal_psd(original_signal, sampling_rate=eeg_rate, method="welch")
    psd_filt = nk.signal_psd(filter_signal, sampling_rate=eeg_rate, method="welch")

    # Plot to confirm the frequencies are still present after the Butterworth filter,
    plt.figure(figsize=(10, 6))
    plt.semilogy(psd_raw["Frequency"], psd_raw["Power"], label='Raw', alpha=0.5)
    plt.semilogy(psd_filt["Frequency"], psd_filt["Power"], label='Filtered', alpha=0.8)
    plt.xlim(49, 60) # Focus on relevant EEG range
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('PSD (dB/Hz)')
    plt.title('Raw vs Filtered EEG Spectrum')
    plt.legend()
    plt.grid(True)
    plt.show()

    peak_raw = psd_raw[psd_raw["Frequency"] == 50]['Power'].values[0]
    peak_raw_60 = psd_raw[psd_raw["Frequency"] == 60]['Power'].values[0]
    peak_filt = psd_filt[psd_filt["Frequency"] == 50]['Power'].values[0]
    peak_filt_60 = psd_filt[psd_filt["Frequency"] == 60]['Power'].values[0]
    print(f"Power at 50Hz: {peak_raw} (Raw) -> {peak_filt} (Cleaned)")
    print(f"Power at 60Hz: {peak_raw_60} (Raw) -> {peak_filt_60} (Cleaned)")

    print("AFTER THE REMOVAL OF POWER LINE")
    psd_power = nk.signal_psd(cleaned, sampling_rate=eeg_rate, method="welch")

     # Plot to confirm the frequencies are still present after the removal of power line,
    plt.figure(figsize=(10, 6))
    plt.semilogy(psd_raw["Frequency"], psd_raw["Power"], label='Raw', alpha=0.5)
    plt.semilogy(psd_filt["Frequency"], psd_filt["Power"], label='Filtered', alpha=0.8)
    plt.semilogy(psd_power["Frequency"], psd_power["Power"], label='After Power Line Removal', alpha=0.8)
    plt.xlim(49, 60) # Focus on relevant EEG range
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('PSD (dB/Hz)')
    plt.title('Raw vs Filtered EEG Spectrum (after power line removal)')
    plt.legend()
    plt.grid(True)
    plt.show()

    peak_power = psd_power[psd_power["Frequency"] == 50]['Power'].values[0]
    peak_power_60 = psd_power[psd_power["Frequency"] == 60]['Power'].values[0]
    print(f"Power at 50Hz: {peak_raw} (Raw) -> {peak_filt} (Cleaned) -> {peak_power} (After Power Line Removal)")
    print(f"Power at 60Hz: {peak_raw_60} (Raw) -> {peak_filt_60} (Cleaned) -> {peak_power_60} (After Power Line Removal)")

    nk.signal_plot([original_signal, eeg_detrended], sampling_rate=eeg_rate, labels =['Raw', 'Cleaned & Detrended'])

    psd_detrended = nk.signal_psd(eeg_detrended, sampling_rate=eeg_rate, method="welch")

    plt.figure(figsize=(10, 6))
    plt.semilogy(psd_raw["Frequency"], psd_raw["Power"], label='Raw', alpha=0.5)
    plt.semilogy(psd_power["Frequency"], psd_filt["Power"], label='Filtered', alpha=0.8)
    plt.semilogy(psd_detrended["Frequency"], psd_detrended["Power"], label='Detrended', alpha=0.8)
    plt.xlim(49, 60) # Focus on relevant EEG range
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('PSD (dB/Hz)')
    plt.title('Raw vs Filtered EEG Spectrum (after power line removal) vs detrended')
    plt.legend()
    plt.grid(True)
    plt.show()

    peak_detrended = psd_detrended[psd_detrended["Frequency"] == 50]['Power'].values[0]
    peak_detrended_60 = psd_detrended[psd_detrended["Frequency"] == 60]['Power'].values[0]
    print(f"Power at 50Hz: {peak_raw} (Raw) -> {peak_filt} (Cleaned) -> {peak_power} (After Power Line Removal) -> {peak_detrended} (Detrended)")
    print(f"Power at 60Hz: {peak_raw_60} (Raw) -> {peak_filt_60} (Cleaned) -> {peak_power_60} (After Power Line Removal) -> {peak_detrended_60} (Detrended)")



### Remove all EEG artifacts for all stimuli sample in one channel

In [ ]:
def remove_eeg_artifacts_stimuli_one_channel(eeg_rate, eeg_stimuli, ichannel):
   # print(eeg_stimuli.shape) # (12800,) 12800 samples for the first video clip
  #  print(eeg_stimuli[:, ichannel].shape) # print the samples of the first channel
    samples_one_channel = eeg_stimuli[:, ichannel]

    #print("THIS IS RAW SIGNAL FOR THE FIRST CHANNEL OF ONE VIDEO")
    #nk.signal_plot(samples_one_channel, sampling_rate=eeg_rate)

    # Apply bandpass filter (e.g., 0.5-45 Hz)
    cleaned_eeg = nk.signal_filter(samples_one_channel, sampling_rate=eeg_rate, lowcut=0.5, highcut=45, method="butterworth", order=4)

    # Removes 50Hz or 60Hz interference
    cleaned = nk.signal_filter(cleaned_eeg, sampling_rate=eeg_rate, method="powerline", powerline=60)


   # print("AFTER THE APPLICATION OF BUTTERWORTH FILTER AND REMOVED POWER LINE")
    #nk.signal_plot([samples_one_channel, cleaned], sampling_rate=eeg_rate, labels =['Raw', 'Cleaned'])

    # Remove linear trend from the cleaned EEG signal
    eeg_detrended = nk.signal_detrend(cleaned, method="polynomial", order=1)

    #Plot PSD to confirm the frequencies are not present after the Butterworth filter and power line removal, and after detrending
   # plot_filter_psd_signal_eeg(samples_one_channel, cleaned_eeg, cleaned, eeg_detrended, eeg_rate)

    return eeg_detrended




### Remove all baseline in EEG signals of one channel

In [ ]:
def remove_eeg_baseline_one_channel(eeg_rate, eeg_baseline, cleaned_eeg, ichannel, ivideo):

    baseline_signal = eeg_baseline[0, 0][ivideo,0][:, ichannel] # the baseline signal for the first video clip and one channel

    # Calculate the mean of the baseline signal
    baseline_mean = np.mean(baseline_signal) # Calculate the mean across all samples in the baseline for one channel

    corrected_eeg = cleaned_eeg - baseline_mean # Subtract the baseline signal from the cleaned EEG signal to remove the baseline

  #  nk.signal_plot([cleaned_eeg, corrected_eeg], sampling_rate=eeg_rate, labels =['Cleaned', 'Baseline Corrected'])
    return corrected_eeg


### Remove all EEG artefacts and baseline in EEG for all samples of one video for one participant (all 14 channels)

In [ ]:
def remove_eeg_signal_one_video(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes,iparticipant, ivideo):

    eeg_stimuli_no_artifacts = remove_artifacts_eeg_one_video(eeg_rate, eeg_stimuli[iparticipant], ivideo, eeg_electrodes)

    cleaned_eeg_one_video = []
    number_channels = len(eeg_electrodes)
    for ichannel in range(number_channels):
        baseline = eeg_baseline[iparticipant]
        cleaned_eeg = remove_eeg_artifacts_stimuli_one_channel(eeg_rate, eeg_stimuli_no_artifacts, ichannel)
        eeg_without_baseline = remove_eeg_baseline_one_channel(eeg_rate, baseline, cleaned_eeg, ichannel, ivideo)
        cleaned_eeg_one_video.append(eeg_without_baseline)

    return cleaned_eeg_one_video

### Remove all EEG artefacts and baseline in EEG for all samples of all video (18 videos) for one participant

In [ ]:
def remove_eeg_artifacts_one_participant(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes, iparticipant, num_video_clips):

  cleaned_eeg_signal_one_participant = []

  for ivideo in range(num_video_clips):
    cleaned_eeg_one_video = remove_eeg_signal_one_video(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes, iparticipant, ivideo)
    cleaned_eeg_signal_one_participant.append(cleaned_eeg_one_video)

  return cleaned_eeg_signal_one_participant

### Remove all EEG artefacts in EEG of all videos (18 videos) watched by all participants (23 participants)

In [ ]:
def remove_eeg_artifacts_all_participants(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes, num_participants, num_video_clips):
   cleaned_eeg_signal_all_participants = []

   for iparticipant in range(num_participants):
      cleaned_eeg_signal_one_participant = remove_eeg_artifacts_one_participant(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes, iparticipant, num_video_clips)
      cleaned_eeg_signal_all_participants.append(cleaned_eeg_signal_one_participant)
      print(f"Finished processing participant {iparticipant}")
      print(cleaned_eeg_signal_all_participants[iparticipant]) # print the cleaned EEG signal of all video clips for one participant after artifact removal and baseline correction
      print("================================================================================================")

   return cleaned_eeg_signal_all_participants

### Extract EEG features for one channel (subject independent)

In [ ]:
def extract_eeg_features_one_channel(cleaned_eeg, eeg_rate, ichannel):
    alpha_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Alpha'], method="multitaper", relative = True) # Extract the power of the alpha band for each epoch
    beta_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Beta'], method="multitaper", relative = True) # Extract the power of the beta band for each epoch
    theta_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Theta'], method="multitaper", relative = True) # Extract the power of the theta band for each epoch
    delta_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Delta'], method="multitaper", relative = True) # Extract the power of the delta band for each epoch
    gamma_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Gamma'], method="multitaper", relative = True) # Extract the power of the gamma band for each epoch
 #   alpha_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Alpha'], method="welch") # Extract the power of the alpha band for each epoch
  #  beta_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Beta'], method="welch") # Extract the power of the beta band for each epoch
  #  theta_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Theta'], method="welch") # Extract the power of the theta band for each epoch
  #  delta_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Delta'], method="welch") # Extract the power of the delta band for each epoch
  #  gamma_power = nk.eeg_power(cleaned_eeg, sampling_rate=eeg_rate, frequency_band=['Gamma'], method="welch") # Extract the power of the gamma band for each epoch

    alpha_feature = alpha_power["Alpha"].values[0]
    beta_feature = beta_power["Beta"].values[0]
    theta_feature = theta_power["Theta"].values[0]
    delta_feature = delta_power["Delta"].values[0]
    gamma_feature = gamma_power["Gamma"].values[0]

    eeg_features_one_channel = [alpha_feature, beta_feature, theta_feature] # Combine the alpha, beta, theta, delta, and gamma power features into a single array for one channel of one video clip
    titles = ["eeg_alpha_", "eeg_beta_", "eeg_theta_"]
    for iband in range(len(titles)):
        titles[iband] = titles[iband] + str(ichannel)

    return eeg_features_one_channel, titles


### Extract EEG features for all 14 channels of one video watched by one participant without VAD labels

In [ ]:
def extract_eeg_features_one_video(cleaned_eeg, eeg_rate, ivideo):
    number_channels = len(cleaned_eeg)

    eeg_features_all_channels = []
    eeg_titles_all_channels = []

    for ichannel in range(number_channels):
        eeg_features_one_channel, titles = extract_eeg_features_one_channel(cleaned_eeg[ichannel], eeg_rate, ichannel)
        eeg_features_all_channels.extend(eeg_features_one_channel)
        eeg_titles_all_channels.extend(titles)

    return eeg_features_all_channels, eeg_titles_all_channels


### Extract EEG features for all 14 channels of one video watched by one participant with VAD labels

In [ ]:
def extract_eeg_features_one_video_with_vad(cleaned_eeg, eeg_rate, ivideo, iparticipant, valences, arousals, dominances):
  eeg_features, title = extract_eeg_features_one_video(cleaned_eeg, eeg_rate, ivideo)

  title.extend(["valence", "arousal", "dominance", "video ID"])
  valence = valences[iparticipant][ivideo]
  arousal = arousals[iparticipant][ivideo]
  dominance = dominances[iparticipant][ivideo]

  vad_labels = [valence, arousal, dominance, ivideo]
  eeg_features.extend(vad_labels)

  return eeg_features, title

### Extract EEG features for all videos (18 videos) watched by one participant

In [ ]:
def extract_eeg_features_one_participant(cleaned_eeg, eeg_rate, iparticipant, num_video_clips):

    eeg_features_one_participant, titles = extract_eeg_features_one_video(cleaned_eeg[0], eeg_rate, 0)
    eeg_titles = titles
    eeg_titles.extend(["video ID", "participant"])

    video_id = []
    for ivideo in range(1, num_video_clips):
        eeg_features_one_video, titles = extract_eeg_features_one_video(cleaned_eeg[ivideo], eeg_rate, ivideo)
        eeg_features_one_participant = np.vstack([eeg_features_one_participant, eeg_features_one_video]) # Combine the features of all video clips into a single array for one participant
        video_id.append(ivideo)

    eeg_features_one_participant = np.column_stack([eeg_features_one_participant, video_id]) # Add the video IDs as the last column of the features array for one participant
    eeg_features_one_participant = np.column_stack([eeg_features_one_participant, [iparticipant]*eeg_features_one_participant.shape[0]]) # Add the participant index as the last column of the features array for one participant

    print(f"Extracted features for participant {iparticipant}: {eeg_features_one_participant.shape}") # Print the shape of the extracted features for one participant
    print(eeg_features_one_participant) # Print the extracted features for one participant

    return eeg_features_one_participant, eeg_titles


### Extract EEG features for all videos (18 videos) watched by one participant with VAD labels

In [ ]:
def extract_eeg_features_one_participant_with_vad(cleaned_eeg, eeg_rate, iparticipant, num_video_clips, valences, arousals, dominances):

   eeg_features_one_participant, titles = extract_eeg_features_one_video_with_vad(cleaned_eeg[0], eeg_rate, 0, iparticipant, valences, arousals, dominances)
   eeg_titles = titles
   eeg_titles.extend(["participant"])

   for ivideo in range(1, num_video_clips):
        eeg_features_one_video, titles = extract_eeg_features_one_video_with_vad(cleaned_eeg[ivideo], eeg_rate, ivideo, iparticipant, valences, arousals, dominances)
        eeg_features_one_participant = np.vstack([eeg_features_one_participant, eeg_features_one_video]) # Combine the features of all video clips into a single array for one participant

   eeg_features_one_participant = np.column_stack([eeg_features_one_participant, [iparticipant]*eeg_features_one_participant.shape[0]]) # Add the participant index as the last column of the features array for one participant

   print(f"Extracted features for participant {iparticipant}: {eeg_features_one_participant.shape}") # Print the shape of the extracted features for one participant
   print(eeg_features_one_participant) # Print the extracted features for one participant

   return eeg_features_one_participant, eeg_titles

### Extract EEG features for all videos watched by all participants (23 participants)

In [ ]:
def extract_eeg_features_all_participants(cleaned_eeg, eeg_rate, num_participants, num_video_clips):

    eeg_features_all_participants = []

    for iparticipant in range(num_participants):
        eeg_features_one_participant, titles = extract_eeg_features_one_participant(cleaned_eeg[iparticipant], eeg_rate, iparticipant, num_video_clips)
        eeg_features_all_participants.append(eeg_features_one_participant)

    eeg_features_all_participants = np.vstack(eeg_features_all_participants) # Combine the features of all participants into a single array
    titles_all_participants = titles

    return eeg_features_all_participants, titles_all_participants

### Extract EEG features for all videos watched by all participants with VAD (23 participants)

In [ ]:
def extract_eeg_features_all_participants_with_vad(cleaned_eeg, eeg_rate, num_participants, num_video_clips, valences, arousals, dominances):

    eeg_features_all_participants = []

    for iparticipant in range(num_participants):
        eeg_features_one_participant, titles = extract_eeg_features_one_participant_with_vad(cleaned_eeg[iparticipant], eeg_rate, iparticipant, num_video_clips, valences, arousals, dominances)
        eeg_features_all_participants.append(eeg_features_one_participant)

    eeg_features_all_participants = np.vstack(eeg_features_all_participants) # Combine the features of all participants into a single array
    titles_all_participants = titles

    return eeg_features_all_participants, titles_all_participants

### Plot the PSD signal and signal graphs of one channel in ECG

In [ ]:
def plot_ecg_signal(ecg_signal, cleaned_ecg, ecg_rate, info):
   nk.signal_plot(ecg_signal, ecg_rate)

   nk.ecg_plot(cleaned_ecg, info)

   nk.signal_plot(cleaned_ecg[['ECG_Raw','ECG_Clean']], ecg_rate)

   psd_raw = nk.signal_psd(ecg_signal, ecg_rate, method="welch")
   psd_filt = nk.signal_psd(cleaned_ecg['ECG_Clean'], ecg_rate, method="welch")

   # Plot to confirm the frequencies are still present after the Butterworth filter,
   plt.figure(figsize=(10, 6))
   plt.semilogy(psd_raw["Frequency"], psd_raw["Power"], label='Raw', alpha=0.5)
   plt.semilogy(psd_filt["Frequency"], psd_filt["Power"], label='cleaned', alpha=0.8)
   plt.xlim(49, 60) # Focus on relevant ECG range
   plt.xlabel('Frequency (Hz)')
   plt.ylabel('PSD (dB/Hz)')
   plt.title('Raw vs Filtered ECG Spectrum')
   plt.legend()
   plt.grid(True)
   plt.show()

   # Find the index of the frequency closest to 50Hz
   idx_closest_50 = (psd_raw["Frequency"] - 50).abs().idxmin()
   # Get the Power value at that index
   peak_raw = psd_raw.loc[idx_closest_50, 'Power']

   idx_closest_60 = (psd_raw["Frequency"] - 60).abs().idxmin()
   peak_raw_60 = psd_raw.loc[idx_closest_60, 'Power']

   # Find the index of the frequency closest to 50Hz in the filtered PSD
   idx_closest_filt = (psd_filt["Frequency"] - 50).abs().idxmin()
   # Safely extract the Power value
   peak_filt = psd_filt.loc[idx_closest_filt, 'Power']

   idx_closest_filt_60 = (psd_filt["Frequency"] - 60).abs().idxmin()
   peak_filt_60 = psd_filt.loc[idx_closest_filt_60, 'Power']

   print(f"Power at 50Hz: {peak_raw} (Raw) -> {peak_filt} (Cleaned)")
   print(f"Power at 60Hz: {peak_raw_60} (Raw) -> {peak_filt_60} (Cleaned)")


### Plot graph to confirm that low quality frequency signals are filtered out

In [ ]:
def plot_ecg_signal_without_low_quality_segments(cleaned_ecg_stimuli, cleaned_ecg_without_low_segment, threshold):
     # 1. Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), sharex=True)

    # 2. Plot the original cleaned signal and highlight the Quality threshold
    ax1.plot(cleaned_ecg_stimuli["ECG_Clean"], label="Full Signal", color="lightgray", alpha=0.5)
    ax1.plot(cleaned_ecg_stimuli["ECG_Quality"], label="Quality Score", color="blue", linestyle='--')
    ax1.axhline(y=threshold, color='red', linestyle='-', label="Threshold " + str(threshold))
    ax1.set_title("Original Signal & Quality Score")
    ax1.legend(loc="upper right")

    # 3. Plot the filtered signal (this will have gaps where quality was < 0.5)
    ax2.plot(cleaned_ecg_without_low_segment["ECG_Clean"], label="High Quality Segments Only", color="green")
    ax2.set_title("Final Filtered Signal (Proven High Quality)")
    ax2.set_xlabel("Samples")
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.show()


### Remove artefacts in one channel of ECG stimulated by watching one video (only two channels - one for left and one for right)

In [ ]:
def preprocess_ecg_one_channel(ecg_stimuli, ecg_baseline, ecg_rate, ivideo, ichannel):

    ecg_stimuli_signal = ecg_stimuli[0][0, 0][ivideo, 0][:, ichannel] # the ECG signal for one video clip and one channel
    ecg_baseline_signal = ecg_baseline[0][0, 0][ivideo, 0][:, ichannel]

    ecg_stimuli_signal = ecg_stimuli_signal.astype(float)
    ecg_baseline_signal = ecg_baseline_signal.astype(float)

    #clean the noise in ECG signal
    cleaned_ecg_stimuli, ecg_stimuli_info = nk.ecg_process(ecg_stimuli_signal, ecg_rate, method="elgendi2010", correct_artifacts=True)
    cleaned_ecg_baseline, ecg_baseline_info = nk.ecg_process(ecg_baseline_signal, ecg_rate, method="elgendi2010", correct_artifacts=True)

    #Apply threshold to filter low-quality signals and baselines
    cleaned_ecg_without_low_segment = cleaned_ecg_stimuli[cleaned_ecg_stimuli['ECG_Quality'] > 0.5]
    cleaned_ecg_baseline_without_low_segment = cleaned_ecg_baseline[cleaned_ecg_baseline['ECG_Quality'] > 0.7]


    return cleaned_ecg_without_low_segment, cleaned_ecg_baseline_without_low_segment





### Extract features for ECG stimuli and baseline of one channel

In [ ]:
def extract_features_ecg_channel(cleaned_ecg, cleaned_ecg_baseline, ecg_rate):

    # Extract Interval-related features (e.g., mean HR, QRS duration)
    interval_features = nk.ecg_intervalrelated(cleaned_ecg, ecg_rate)
    baseline_features = nk.ecg_intervalrelated(cleaned_ecg_baseline, ecg_rate)

    # 1. Flatten the nested arrays into simple floats
    # np.squeeze removes the extra array brackets []
    flat_interval_features = interval_features.map(lambda x: np.squeeze(x) if isinstance(x, (list, np.ndarray)) else x)
    flat_baseline_features = baseline_features.map(lambda x: np.squeeze(x) if isinstance(x, (list, np.ndarray)) else x)

    return flat_interval_features, flat_baseline_features




### Remove all artefacts and extract the features in ECG stimulated by watching one video (two channels)

In [ ]:
def preprocess_ecg_one_video(ecg_stimuli, ecg_baseline, ecg_rate, ivideo):
    left_cleaned_ecg, left_cleaned_ecg_baseline = preprocess_ecg_one_channel(ecg_stimuli, ecg_baseline, ecg_rate, ivideo, 0)
    right_cleaned_ecg, right_cleaned_ecg_baseline = preprocess_ecg_one_channel(ecg_stimuli, ecg_baseline, ecg_rate, ivideo, 1)
    left_ecg_features, left_ecg_baseline_features = extract_features_ecg_channel(left_cleaned_ecg, left_cleaned_ecg_baseline, ecg_rate)
    right_ecg_features, right_ecg_baseline_features = extract_features_ecg_channel(right_cleaned_ecg, right_cleaned_ecg_baseline, ecg_rate)


    left_ecg_features_prefix = left_ecg_features.add_prefix("left_")
    left_ecg_features = left_ecg_features_prefix
    left_ecg_baseline_prefix = left_ecg_baseline_features.add_prefix("left_")
    left_ecg_baseline_features = left_ecg_baseline_prefix
    right_ecg_features_prefix = right_ecg_features.add_prefix("right_")
    right_ecg_features = right_ecg_features_prefix
    right_ecg_baseline_prefix = right_ecg_baseline_features.add_prefix("right_")
    right_ecg_baseline_features = right_ecg_baseline_prefix

    return left_ecg_features, left_ecg_baseline_features, right_ecg_features, right_ecg_baseline_features


### Remove all artefacts and extract features in ECG stimulated by watching videos of one participant (18 videos)

In [ ]:
def preprocess_ecg_one_participant(ecg_stimuli, ecg_baseline, ecg_rate, iparticipant, num_video_clips):

    ecg_stimuli_participant = pd.DataFrame()
    ecg_baseline_participant = pd.DataFrame()

    for ivideo in range(num_video_clips):
        left_cleaned_ecg, left_ecg_baseline, right_cleaned_ecg, right_ecg_baseline = preprocess_ecg_one_video(ecg_stimuli, ecg_baseline, ecg_rate, ivideo)

        ecg_stimuli_video = pd.concat([left_cleaned_ecg, right_cleaned_ecg], axis = 1)
        ecg_baseline_video = pd.concat([left_ecg_baseline, right_ecg_baseline], axis = 1)

        ecg_stimuli_participant = pd.concat([ecg_stimuli_participant, ecg_stimuli_video], axis = 0)
        ecg_baseline_participant = pd.concat([ecg_baseline_participant, ecg_baseline_video], axis = 0)

    return ecg_stimuli_participant, ecg_baseline_participant


### Remove all artefacts and extract features in ECG stimulated by watching videos of one participant with VAD (18 videos)

In [ ]:
def preprocess_ecg_one_participant_vad(ecg_stimuli, ecg_baseline, ecg_rate, iparticipant, num_video_clips, valences, arousals, dominances):
    ecg_stimuli_participant, ecg_baseline_participant = preprocess_ecg_one_participant(ecg_stimuli, ecg_baseline, ecg_rate, iparticipant, num_video_clips)

    valence_vals = []
    arousal_vals = []
    dominance_vals = []
    video_ids = []
    for ivideo in range(num_video_clips):
        valence_vals.append(valences[iparticipant][ivideo])
        arousal_vals.append(arousals[iparticipant][ivideo])
        dominance_vals.append(dominances[iparticipant][ivideo])
        video_ids.append(ivideo)

    ecg_stimuli_participant['valence'] = valence_vals
    ecg_stimuli_participant['arousal'] = arousal_vals
    ecg_stimuli_participant['dominance'] = dominance_vals
    ecg_stimuli_participant['participant'] = iparticipant
    ecg_stimuli_participant['video ID'] = video_ids
    ecg_baseline_participant['valence'] = valence_vals
    ecg_baseline_participant['arousal'] = arousal_vals
    ecg_baseline_participant['dominance'] = dominance_vals
    ecg_baseline_participant['participant'] = iparticipant
    ecg_baseline_participant['video ID'] = video_ids

    return ecg_stimuli_participant, ecg_baseline_participant

### Remove all artifaces and extract features in ECG stimulated by watching videos of all 23 participants

In [ ]:
def preprocess_extract_ecg_features(ecg_stimuli, ecg_baseline, ecg_rate, num_participants, num_video_clips):
    ecg_stimuli_features = pd.DataFrame()
    ecg_baseline_features = pd.DataFrame()

    for iparticipant in range(num_participants):
        ecg_stimuli_participant_feature, ecg_baseline_participant_feature = preprocess_ecg_one_participant(ecg_stimuli, ecg_baseline, ecg_rate, iparticipant, num_video_clips)
        ecg_stimuli_features = pd.concat([ecg_stimuli_features, ecg_stimuli_participant_feature], axis = 0)
        ecg_baseline_features = pd.concat([ecg_baseline_features, ecg_baseline_participant_feature], axis = 0)

    return ecg_stimuli_features, ecg_baseline_features

### Remove all artifacts and extract features in ECG with VAD stimulated by watching videos of all 23 participants

In [ ]:
def preprocess_extract_ecg_features_vad(ecg_stimuli, ecg_baseline, ecg_rate, num_participants, num_video_clips, valences, arousals, dominances):
    ecg_stimuli_features = pd.DataFrame()
    ecg_baseline_features = pd.DataFrame()

    for iparticipant in range(num_participants):
        ecg_stimuli_participant_feature, ecg_baseline_participant_feature = preprocess_ecg_one_participant_vad(ecg_stimuli, ecg_baseline, ecg_rate, iparticipant, num_video_clips, valences, arousals, dominances)
        ecg_stimuli_features = pd.concat([ecg_stimuli_features, ecg_stimuli_participant_feature], axis = 0)
        ecg_baseline_features = pd.concat([ecg_baseline_features, ecg_baseline_participant_feature], axis = 0)

    return ecg_stimuli_features, ecg_baseline_features

### Clean up NaN values in both stimuli and baseline features for all participants

In [ ]:
def remove_nan_features_samples(ecg_stimuli_features, ecg_baseline_features, eeg_features, eeg_title):

    feature_threshold = 0.3

    eeg_features_df = pd.DataFrame(eeg_features, columns = eeg_title)

    print(eeg_features_df)

    ecg_stimuli_total_nan_counts = ecg_stimuli_features.isna().sum()
    ecg_stimuli_nan_percentage =  ecg_stimuli_total_nan_counts / len(ecg_stimuli_features)
    ecg_baseline_total_nan_counts = ecg_baseline_features.isna().sum()
    ecg_baseline_nan_percentage =  ecg_baseline_total_nan_counts / len(ecg_baseline_features)
    eeg_total_nan_counts = eeg_features_df.isna().sum()
    eeg_nan_percentage =  eeg_total_nan_counts / len(eeg_features_df)
   
    ecg_stimuli_columns_to_drop = ecg_stimuli_features.columns[ecg_stimuli_nan_percentage >= feature_threshold].tolist()
    ecg_stimuli_features_cleaned = ecg_stimuli_features.drop(columns = ecg_stimuli_columns_to_drop)
    ecg_baseline_columns_to_drop = ecg_baseline_features.columns[ecg_baseline_nan_percentage >= feature_threshold].tolist()
    ecg_baseline_features_cleaned = ecg_baseline_features.drop(columns = ecg_baseline_columns_to_drop)
    eeg_columns_to_drop = eeg_features_df.columns[eeg_nan_percentage >= feature_threshold].tolist()
    eeg_features_cleaned = eeg_features_df.drop(columns = eeg_columns_to_drop)

    #merge the cleaned features of ECG stimuli, ECG baseline, and EEG into a single dataset for further analysis
    num_ecg_stimuli_features = ecg_stimuli_features_cleaned.shape[1]
    num_ecg_baseline_features = ecg_baseline_features_cleaned.shape[1]

    combined_features = pd.concat([ecg_stimuli_features_cleaned.reset_index(drop=True), ecg_baseline_features_cleaned.reset_index(drop=True), eeg_features_cleaned.reset_index(drop=True)], axis = 1)
   
    sample_threshold = int(combined_features.shape[1] * 0.9) # Set the threshold for the percentage of NaN values per sample (e.g., 10% of total features)
  
    #filter the whole dataset (e.g keep rows with <= 30% NaNs)
    combined_features_cleaned = combined_features.dropna(thresh = sample_threshold)
  
    #split the combined data frame to individual datasets for ECG stimuli, ECG baseline, and EEG features
    cleaned_ecg_stimuli_features = combined_features_cleaned.iloc[:, :num_ecg_stimuli_features]
    cleaned_ecg_baseline_features = combined_features_cleaned.iloc[:, num_ecg_stimuli_features : num_ecg_stimuli_features + num_ecg_baseline_features]
    cleaned_eeg_features = combined_features_cleaned.iloc[:, num_ecg_stimuli_features + num_ecg_baseline_features :]

    ecg_stimuli_total_nan_counts = cleaned_ecg_stimuli_features.isnull().values.any()
    ecg_baseline_total_nan_counts = cleaned_ecg_baseline_features.isnull().values.any()
    eeg_total_nan_counts = cleaned_eeg_features.isnull().values.any()
  
    #impute the remaining NaN values with the mean of each feature column by participant (subject-independent imputation)
    if ecg_stimuli_total_nan_counts:
        feature_columns = [col for col in cleaned_ecg_stimuli_features.columns if col not in ['participant', 'valence', 'arousal', 'dominance', 'video ID']]
        cleaned_ecg_stimuli_features[feature_columns] = cleaned_ecg_stimuli_features.groupby('participant')[feature_columns].transform(lambda x: x.fillna(x.mean()))
    if ecg_baseline_total_nan_counts:
        valid_columns = [col for col in cleaned_ecg_baseline_features.columns if col  not in ['participant', 'valence', 'arousal', 'dominance', 'video ID']] # Ensure we only impute columns that exist in the baseline features    
        cleaned_ecg_baseline_features[valid_columns] = cleaned_ecg_baseline_features.groupby('participant')[valid_columns].transform(lambda x: x.fillna(x.mean()))
    if eeg_total_nan_counts:
        eeg_cols = [col for col in cleaned_eeg_features.columns if col not in ['participant', 'valence', 'arousal', 'dominance', 'video ID']]
        cleaned_eeg_features[eeg_cols] = cleaned_eeg_features.groupby('participant')[eeg_cols].transform(lambda x: x.fillna(x.mean()))
      
    return cleaned_ecg_stimuli_features, cleaned_ecg_baseline_features, cleaned_eeg_features



### Detect and replace infinity values with nan values in both stimuli and baseline ECG features for all participants

In [ ]:
def detect_infinity_featuress_samples(ecg_stimuli_features, ecg_baseline_features):

    feature_columns = [col for col in ecg_stimuli_features.columns if col not in ['participant', 'valence', 'arousal', 'dominance', 'video ID']]
    # Check for infinite values in the features DataFrames
    infinity_in_stimuli = ecg_stimuli_features.groupby(['participant'])[feature_columns].apply(lambda group: np.isinf(group[feature_columns]).any(axis = 1).sum())

    baseline_columns = [col for col in ecg_baseline_features.columns if col not in ['participant', 'valence', 'arousal', 'dominance', 'video ID']]
    infinity_in_baseline = ecg_baseline_features.groupby(['participant'])[baseline_columns].apply(lambda group: np.isinf(group[baseline_columns]).any(axis = 1).sum())


    if infinity_in_stimuli.sum() > 0:
        
        #calculate the mean per participant (ignoring Infs and NaNs)
        #We replace the infinity values with NaN, then calculate the mean for each participant and video ID, and finally fill the NaN values with the corresponding participant-video mean  
        participant_mean = ecg_stimuli_features.replace([np.inf, -np.inf], np.nan).groupby(['participant']).transform('mean')   
        ecg_stimuli_features[feature_columns] = ecg_stimuli_features[feature_columns].mask(np.isinf(ecg_stimuli_features[feature_columns])).fillna(participant_mean[feature_columns])
        
    if infinity_in_baseline.sum() > 0:
        participant_mean_baseline = ecg_baseline_features.replace([np.inf, -np.inf], np.nan).groupby(['participant']).transform('mean')   
        ecg_baseline_features[baseline_columns] = ecg_baseline_features[baseline_columns].mask(np.isinf(ecg_baseline_features[baseline_columns])).fillna(participant_mean_baseline[baseline_columns])
      
    return ecg_stimuli_features, ecg_baseline_features    

### Remove baseline for one ECG channel by subtracting the baseline HRV metrics from the stimuli HRV metrics

In [ ]:
def ecg_baseline_removal(ecg_stimuli_features, ecg_baseline_features):

    ecg_stimuli_features_only = ecg_stimuli_features.iloc[:, ~ecg_stimuli_features.columns.isin(['participant', 'valence', 'arousal', 'dominance', 'video ID'])]
    ecg_baseline_features_only = ecg_baseline_features.iloc[:, ~ecg_baseline_features.columns.isin(['participant', 'valence', 'arousal', 'dominance', 'video ID'])]

    #Get the common columns between the two datasets (excluding participant and VAD labels)
    common_columns = ecg_stimuli_features_only.columns.intersection(ecg_baseline_features_only.columns)
   
    #Filter the datasets to keep only the common columns
    ecg_stimuli_features_only = ecg_stimuli_features_only[common_columns]
    ecg_baseline_features_only = ecg_baseline_features_only[common_columns]

    vad_videos = ecg_stimuli_features[['valence', 'arousal', 'dominance', 'video ID', 'participant']]

    # Reset indices to ensure they align by row position (0 to 35)
    ecg_stimuli_features_only = ecg_stimuli_features_only.reset_index(drop=True)
    ecg_baseline_features_only = ecg_baseline_features_only.reset_index(drop=True)

    ecg_features = ecg_stimuli_features_only[common_columns] - ecg_baseline_features_only[common_columns]
  
    #Identify columns with all values are 0
    zero_columns = ecg_features.columns[(ecg_features == 0).all()]
    #Drop the columns with all values are 0
    ecg_features = ecg_features.drop(columns = zero_columns)

    ecg_features = pd.concat([ecg_features.reset_index(drop=True), vad_videos.reset_index(drop=True)], axis = 1)

    return ecg_features, vad_videos

### Define a per-participant outlier detection function

In [ ]:
def identify_participant_outliers(features):
    #Identify outliers (return a boolean mask: True for outliers, False for non-outliers)
    #Default method is the IQR method, which identifies outliers as values that are below Q1 - 1.5*IQR or above Q3 + 1.5*IQR, where Q1 is the 25th percentile and Q3 is the 75th percentile of the data.
    outliers_mask = nk.find_outliers(features, method="standardize", robust = True, exclude = 3, side = "both")

    return outliers_mask

### Function to apply scipy winsorization to a panda series

In [ ]:
def apply_winsor(feature, limit=0.05):
    return winsorize(feature, limits=(limit, limit))

### Detect and remove all outliers in both EEG features and ECG features

In [ ]:
def remove_outliers_eeg_ecg(ecg_features, eeg_features):
    eeg_features_only = eeg_features.iloc[:, ~eeg_features.columns.isin(['eeg_feature_42', 'eeg_feature_43', 'eeg_feature_44', 'eeg_feature_45', 'eeg_feature_46'])]
    ecg_features_only = ecg_features.iloc[:, ~ecg_features.columns.isin(['valence', 'arousal', 'dominance', 'video ID'])]

    #To check all features at once, we can sum the boolean mask across all feature columns for each sample. If the sum is greater than 0, it means that at least one feature in that sample is an outlier.  
    mask_combined_eeg_outliers = eeg_features_only.groupby('participant').transform(lambda x: identify_participant_outliers(x)).any(axis=1) # This will give a boolean mask where True indicates that at least one feature in that sample is an outlier for that participant
    mask_combined_ecg_outliers = ecg_features_only.groupby('participant').transform(lambda x: identify_participant_outliers(x)).any(axis=1) # This will give a boolean mask where True indicates that at least one feature in that sample is an outlier for that participant

    eeg_features_cleaned = eeg_features_only.groupby('participant').transform(lambda x: apply_winsor(x, limit=0.05)) # Apply the outlier removal for each participant separately
    ecg_features_cleaned = ecg_features_only.groupby('participant').transform(lambda x: apply_winsor(x, limit=0.05)) # Apply the outlier removal for each participant separately

    return ecg_features_cleaned, eeg_features_cleaned

    

### Fuse EEG features and ECG features into one data frame

In [ ]:
def fuse_eeg_ecg_features(ecg_features, eeg_features, vad_videos):
        
    #Clean the eeg and ecg dataframes to keep only the feature columns (excluding participant and VAD labels) for fusion
    eeg_features_only = eeg_features.drop(columns = ['valence', 'arousal', 'dominance', 'video ID'])

    eeg_with_ids = pd.concat([eeg_features_only.reset_index(drop=True), vad_videos[['participant', 'video ID']].reset_index(drop=True)], axis = 1) # Add participant and video ID columns back to the EEG features for merging
    ecg_with_ids = pd.concat([ecg_features.reset_index(drop=True), vad_videos[['participant', 'video ID']].reset_index(drop=True)], axis = 1) # Add participant and video ID columns back to the ECG features for merging
  
    #merge the cleaned features of ECG and EEG into a single dataset for further analysis (inner join on participant and video ID)
    fused_features_without_vad = pd.merge(eeg_with_ids, ecg_with_ids, on=['participant', 'video ID'], how='inner')

    #merge the fused features with the VAD labels (inner join on participant and video ID)
    fused_features = pd.merge(fused_features_without_vad, vad_videos[['participant', 'video ID', 'valence', 'arousal', 'dominance']], on=['participant', 'video ID'], how='left')
 
    return fused_features

### Save all cleaned data into a csv file (ready for model prediction)

In [ ]:
def save_data_to_csv(eeg_features, ecg_features, fused_features, vad_videos, output_dir, filename, include_eeg = True, include_ecg = True, include_fused = True):
    # Add this print to verify flags are active
    print(f"Flags: EEG={include_eeg}, ECG={include_ecg}, Fused={include_fused}")
    print(f"Saving files to: {output_dir}")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created output directory: {output_dir}")

    if include_eeg:
        eeg_df = pd.concat([eeg_features.reset_index(drop=True), vad_videos[['participant']].reset_index(drop=True)], axis = 1)
        save_path = os.path.join(output_dir, f"{filename}_eeg_features.csv")
        eeg_df.to_csv(save_path, index=False, encoding='utf-8', header=True)
        print(f"Successfully saved EEG to {save_path}")
    if include_ecg:
        ecg_df = pd.concat([ecg_features.reset_index(drop=True), vad_videos[['valence','arousal','dominance','video ID','participant']].reset_index(drop=True)], axis = 1)
        save_path = os.path.join(output_dir, f"{filename}_ecg_features.csv")
        ecg_df.to_csv(save_path, index=False, encoding='utf-8', header=True)
        print(f"Successfully saved ECG to {save_path}")
    if include_fused:
        end_cols = ['video ID', 'participant']
        other_cols = [col for col in fused_features.columns if col not in end_cols]
        fused_features = fused_features[other_cols + end_cols]
        save_path = os.path.join(output_dir, f"{filename}_fused_features.csv")
        fused_features.to_csv(save_path, index=False, encoding='utf-8', header=True)
        print(f"Successfully saved Fused features to {save_path}")

### Define a main method

In [ ]:
def main():
    #dir_path = "/content/drive/MyDrive/dataset/"
    dir_path = "C:/Users/seanl/Desktop/DREAMER_EXPERIMENT/"
    file_name = "DREAMER.mat"
    dreamer_raw_data, dreamer_raw = load_data_file(dir_path, file_name)

    print("Data loaded successfully. Shape of the raw data:", dreamer_raw_data)
    print(dreamer_raw)

    num_participants, num_video_clips = extract_participants_and_clips(dreamer_raw)
    print(f"Number of participants: {num_participants}")
    print(f"Number of video clips: {num_video_clips}")

    eeg_rate, ecg_rate, eeg_electrodes = extract_signal_information(dreamer_raw)

    print(f"EEG Sampling Rate: {eeg_rate} Hz")
    print(f"ECG Sampling Rate: {ecg_rate} Hz")
    print(f"EEG Electrodes: {eeg_electrodes}")

    eeg_baseline, eeg_stimuli, ecg_baseline, ecg_stimuli = extact_physiological_signal_all_participants(dreamer_raw_data, num_participants)

    valences, arousals, dominances = extract_vad_all_participants(dreamer_raw_data, num_participants, num_video_clips)

    cleaned_eeg_signal = remove_eeg_artifacts_all_participants(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes, num_participants, num_video_clips)
  #  cleaned_eeg_signal = remove_eeg_artifacts_all_participants(eeg_rate, eeg_stimuli, eeg_baseline, eeg_electrodes, 7, 7)
  
    eeg_features, eeg_title = extract_eeg_features_all_participants_with_vad(cleaned_eeg_signal, eeg_rate, num_participants, num_video_clips, valences, arousals, dominances)
   # eeg_features, eeg_title = extract_eeg_features_all_participants_with_vad(cleaned_eeg_signal, eeg_rate, 7, 7, valences, arousals, dominances)
   
    print(eeg_features)
    print(len(eeg_features))
    print(eeg_title)

    ecg_stimuli_features, ecg_baseline_features = preprocess_extract_ecg_features_vad(ecg_stimuli, ecg_baseline, ecg_rate, num_participants, num_video_clips, valences, arousals, dominances)
   # ecg_stimuli_features, ecg_baseline_features = preprocess_extract_ecg_features_vad(ecg_stimuli, ecg_baseline, ecg_rate, 7, 7, valences, arousals, dominances)
   
    print(ecg_stimuli_features)
    print(len(ecg_stimuli_features))
    print("BASELINE FEATURES")
    print(ecg_baseline_features)
    print(len(ecg_baseline_features))
 
    cleaned_ecg_stimuli_features, cleaned_ecg_baseline_features, cleaned_eeg_features = remove_nan_features_samples(ecg_stimuli_features, ecg_baseline_features, eeg_features, eeg_title)

    print(cleaned_ecg_stimuli_features)
    print(cleaned_ecg_stimuli_features.shape)
    print(cleaned_ecg_baseline_features)
    print(cleaned_ecg_baseline_features.shape)
    print(cleaned_eeg_features)
    print(cleaned_eeg_features.shape)

    cleaned_ecg_stimuli_features, cleaned_ecg_baseline_features = detect_infinity_featuress_samples(cleaned_ecg_stimuli_features, cleaned_ecg_baseline_features)
    print(cleaned_ecg_stimuli_features)
    print(cleaned_ecg_stimuli_features.shape)
    print(cleaned_ecg_baseline_features)
    print(cleaned_ecg_baseline_features.shape)

    ecg_features, vad_videos = ecg_baseline_removal(cleaned_ecg_stimuli_features, cleaned_ecg_baseline_features)

    print(ecg_features)
    print(ecg_features.shape)
    print(vad_videos)
    print(vad_videos.shape)

    ecg_features_cleaned, eeg_features_cleaned = remove_outliers_eeg_ecg(ecg_features, cleaned_eeg_features)
    print(eeg_features_cleaned)
    print(eeg_features_cleaned.shape)
    print(ecg_features_cleaned)
    print(ecg_features_cleaned.shape)

    eeg_ecg_df = fuse_eeg_ecg_features(ecg_features_cleaned, eeg_features_cleaned, vad_videos)
    print(eeg_ecg_df.to_string())
    print(eeg_ecg_df.shape)
    print(eeg_ecg_df.columns)

    #output_dir = "/content/drive/MyDrive/dataset/subject_independent"
    output_dir = "C:/Users/seanl/Desktop/DREAMER_EXPERIMENT/"
    save_data_to_csv(eeg_features_cleaned, ecg_features_cleaned, eeg_ecg_df, vad_videos, output_dir, "processed_dreamer_data", include_eeg = True, include_ecg = True, include_fused = True)

if __name__=="__main__":
    main()